# 03 — Açık küme KWS: "bilinmeyen" ve "sessizlik" sınıflarıyla yeniden eğitim

**Amaç:** Modelin "bu ses hiçbir komut değil" diyebilmesi. 8 kelime + `_silence_` + `_unknown_` = **10 sınıf**.

**Veri:** Tam Speech Commands v2 (diğer 27 kelime "bilinmeyen", arka plan gürültüleri "sessizlik") + kartın mikrofonuyla alınmış kayıtların (komutlar + 100 "bilinmeyen").

**Model:** DS-CNN (KWS için standart mimari). Girdi aynı (49 × 10 MFCC, `mu`/`sd` kartla aynı), yani kartta sadece model ve sınıf sayısı değişir.

**Dürüst test:** Senin komutlarının her kelimeden son 3'ü ve "bilinmeyen" kayıtlarının son 20'si eğitime hiç girmez.

Çalışma zamanı türü: **GPU**. Hücreleri yukarıdan aşağı sırayla çalıştır. Toplam ~20–30 dk.

## 1. Dosyaları yükle
`kws_egitim.zip` (komut klasörleri + `unknown`), `mfcc_consts.h` (kartta kullanılan) ve `kws_int8_v2.tflite` (karşılaştırma için) — üçünü birden seç.

In [ ]:
import os, re, pathlib, random, zipfile, shutil, subprocess
import numpy as np, tensorflow as tf
from scipy.io import wavfile
from google.colab import files

for f in ['kws_egitim.zip', 'mfcc_consts.h', 'kws_int8_v2.tflite']:
    if os.path.exists(f): os.remove(f)          # eski kopyalar karışmasın
up = files.upload()
for k in list(up):                               # 'dosya (1).zip' gibi isimleri düzelt
    clean = re.sub(r' \(\d+\)', '', k)
    if clean != k: os.replace(k, clean)
shutil.rmtree('kws_egitim', ignore_errors=True)
with zipfile.ZipFile('kws_egitim.zip') as z:
    z.extractall('.')
for p in sorted(pathlib.Path('kws_egitim').iterdir()):
    if p.is_dir(): print(f'{p.name:8s}: {len(list(p.glob("*.wav")))} kayit')

## 2. Kartta kullanılan normalizasyon değerleri (değişmeyecek)

In [ ]:
h = open('mfcc_consts.h').read()
def c_arr(name):
    body = re.search(r'static const float ' + name + r'\[\d+\] = \{(.*?)\};', h, re.S).group(1)
    return np.array(re.findall(r'(-?\d\.\d+e[+-]\d+)f', body), dtype=np.float32)
mu, sd = c_arr('MFCC_MU'), c_arr('MFCC_SD')
print('mu =', mu.round(2)); print('sd =', sd.round(2))

## 3. Tam Speech Commands v2'yi indir (~2,3 GB, birkaç dakika)

In [ ]:
SC = pathlib.Path('sc2')
if not (SC / 'yes').exists():
    subprocess.run('wget -q -nc http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz', shell=True, check=True)
    SC.mkdir(exist_ok=True)
    subprocess.run('tar -xzf speech_commands_v0.02.tar.gz -C sc2', shell=True, check=True)
print(sorted(p.name for p in SC.iterdir() if p.is_dir()))

## 4. Sınıflar ve resmi eğitim/doğrulama/test bölmesi
Speech Commands'ın kendi listeleri kullanılıyor: aynı konuşmacı hem eğitimde hem testte olmuyor.

In [ ]:
KW = ['down', 'go', 'left', 'no', 'right', 'stop', 'up', 'yes']
LABELS = KW + ['_silence_', '_unknown_']
SIL, UNK = 8, 9
rng = random.Random(0)

val_set  = set(open(SC / 'validation_list.txt').read().split())
test_set = set(open(SC / 'testing_list.txt').read().split())
split_of = lambda rel: 'val' if rel in val_set else 'test' if rel in test_set else 'train'

items = {'train': [], 'val': [], 'test': []}          # (dosya, etiket)
for i, w in enumerate(KW):
    for f in sorted((SC / w).glob('*.wav')):
        items[split_of(f'{w}/{f.name}')].append((f, i))

others = [p.name for p in SC.iterdir() if p.is_dir() and p.name not in KW and not p.name.startswith('_')]
unk = [(f, f'{w}/{f.name}') for w in others for f in sorted((SC / w).glob('*.wav'))]
rng.shuffle(unk)
lim, cnt = {'train': 9000, 'val': 900, 'test': 900}, {'train': 0, 'val': 0, 'test': 0}
for f, rel in unk:
    s = split_of(rel)
    if cnt[s] < lim[s]:
        items[s].append((f, UNK)); cnt[s] += 1

for s in items:
    print(s, len(items[s]), np.bincount([l for _, l in items[s]], minlength=10))

## 5. Ses yükleme, çoğaltma (augmentation) ve toplu MFCC
MFCC, kartla birebir aynı tarif (640/320/1024, 40 mel, 20–4000 Hz, log+1e-6, ilk 10 katsayı).

In [ ]:
def load_wav(path, n=16000):
    _, x = wavfile.read(path)
    x = x.astype(np.float32) / 32768.0
    if len(x) < n: x = np.pad(x, (0, n - len(x)))
    return x[:n]

bg = []
for f in (SC / '_background_noise_').glob('*.wav'):
    _, b = wavfile.read(f); bg.append(b.astype(np.float32) / 32768.0)

def bg_clip():
    b = rng.choice(bg); st = rng.randrange(0, len(b) - 16000)
    return b[st:st + 16000]

def silence_clip():
    return (bg_clip() * rng.uniform(0.0, 1.0)).astype(np.float32)

def augment(x, p_noise=0.8, max_shift=1600):
    s = rng.randint(-max_shift, max_shift)
    y = np.roll(x, s)
    if s > 0: y[:s] = 0
    elif s < 0: y[s:] = 0
    y = y * rng.uniform(0.7, 1.3)
    if rng.random() < p_noise:                       # gerçek arka plan gürültüsü, SNR 5–25 dB
        n = bg_clip()
        ps, pn = np.mean(y ** 2) + 1e-9, np.mean(n ** 2) + 1e-9
        y = y + n * np.sqrt(ps / (pn * 10 ** (rng.uniform(5, 25) / 10)))
    return np.clip(y, -1, 1).astype(np.float32)

MEL_W = tf.signal.linear_to_mel_weight_matrix(40, 513, 16000, 20.0, 4000.0)
def mfcc_batch(X):
    spec = tf.abs(tf.signal.stft(tf.constant(X), frame_length=640, frame_step=320, fft_length=1024))
    mel = tf.tensordot(spec, MEL_W, 1)
    return tf.signal.mfccs_from_log_mel_spectrograms(tf.math.log(mel + 1e-6))[..., :10].numpy()

def build(sources, bs=1024):
    '''sources: [(sesi üreten fonksiyon, etiket)] -> normalize MFCC, etiketler'''
    Xs, ys, buf, lab = [], [], [], []
    for fn, l in sources:
        buf.append(fn()); lab.append(l)
        if len(buf) == bs:
            Xs.append(mfcc_batch(np.stack(buf))); ys += lab; buf, lab = [], []
    if buf:
        Xs.append(mfcc_batch(np.stack(buf))); ys += lab
    X = (np.concatenate(Xs) - mu) / sd
    return X.astype(np.float32), np.array(ys)

## 6. Veri kümelerini oluştur (en uzun hücre, ~5–10 dk)

In [ ]:
N_KW_TEST, N_UNK_TEST = 3, 20
my = pathlib.Path('kws_egitim')
tr, my_te = [], []
for i, w in enumerate(KW):                                     # senin komutların
    fs = sorted((my / w).glob('*.wav'))
    for f in fs[:-N_KW_TEST]:
        tr.append((lambda f=f: load_wav(f), i))
        tr += [(lambda f=f: augment(load_wav(f), p_noise=0.5), i)] * 15
    my_te += [(lambda f=f: load_wav(f), i) for f in fs[-N_KW_TEST:]]
fs = sorted((my / 'unknown').glob('*.wav'))                    # senin "bilinmeyen" kayıtların
for f in fs[:-N_UNK_TEST]:
    tr.append((lambda f=f: load_wav(f), UNK))
    tr += [(lambda f=f: augment(load_wav(f), p_noise=0.5), UNK)] * 10
my_te += [(lambda f=f: load_wav(f), UNK) for f in fs[-N_UNK_TEST:]]

tr += [(lambda f=f: augment(load_wav(f)), l) for f, l in items['train']]
tr += [(silence_clip, SIL)] * 3000
va  = [(lambda f=f: load_wav(f), l) for f, l in items['val']]  + [(silence_clip, SIL)] * 300
te  = [(lambda f=f: load_wav(f), l) for f, l in items['test']] + [(silence_clip, SIL)] * 300
rng.shuffle(tr)

X_tr, y_tr = build(tr); X_va, y_va = build(va); X_te, y_te = build(te); X_my, y_my = build(my_te)
print('egitim', X_tr.shape, np.bincount(y_tr, minlength=10))
print('dogrulama', X_va.shape, ' test', X_te.shape, ' senin test', X_my.shape)

## 7. DS-CNN modeli ve eğitim

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers
tf.keras.utils.set_random_seed(1)
F = 48
inp = keras.Input((49, 10, 1))
x = layers.Conv2D(F, (10, 4), strides=(2, 2), padding='same', use_bias=False)(inp)
x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
for _ in range(4):
    x = layers.DepthwiseConv2D((3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    x = layers.Conv2D(F, 1, use_bias=False)(x)
    x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
out = layers.Dense(len(LABELS), activation='softmax')(x)
model = keras.Model(inp, out)
model.summary()

model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
hist = model.fit(X_tr[..., None], y_tr, validation_data=(X_va[..., None], y_va), epochs=40, batch_size=128,
                 callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
                            keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)])

## 8. int8'e çevir

In [ ]:
rep = X_tr[:500]
def rep_data():
    for k in range(len(rep)):
        yield [rep[k:k+1, ..., None]]
conv = tf.lite.TFLiteConverter.from_keras_model(model)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.representative_dataset = rep_data
conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv.inference_input_type = tf.int8
conv.inference_output_type = tf.int8
new_model = conv.convert()
print(f'model boyutu: {len(new_model)/1024:.1f} KB')

## 9. Değerlendirme: yanlış kabul ve kaçırma
Kartla **aynı karar kuralı**: en iyi tahmin bir komutsa, güveni ≥ eşik ve ilk iki arasındaki fark ≥ 0,40 ise kabul.
- **Kaçırma:** senin komutlarından tanınmayanlar (24 kayıt)
- **Yanlış kabul:** komut olmayan seslerin (senin 20 "bilinmeyen" + veri seti "bilinmeyen" ve "sessizlik") komut sanılma oranı

In [ ]:
def probs_tflite(mb, Xn):
    it = tf.lite.Interpreter(model_content=mb); it.allocate_tensors()
    i, o = it.get_input_details()[0], it.get_output_details()[0]
    s, zp = i['quantization']; so, zo = o['quantization']
    P = []
    for k in range(len(Xn)):
        q = np.clip(np.round(Xn[k:k+1, ..., None] / s + zp), -128, 127).astype(np.int8)
        it.set_tensor(i['index'], q); it.invoke()
        P.append((it.get_tensor(o['index'])[0].astype(np.float32) - zo) * so)
    return np.array(P)

def decide(P, thr, margin=0.40):
    '''kart kuralı: -1 = kabul yok, aksi halde komut indeksi (0..7)'''
    top = np.argsort(P, axis=1)[:, ::-1]
    p1, p2 = P[np.arange(len(P)), top[:, 0]], P[np.arange(len(P)), top[:, 1]]
    ok = (top[:, 0] < 8) & (p1 >= thr) & (p1 - p2 >= margin)
    return np.where(ok, top[:, 0], -1)

def report(name, P_my, P_te, thr):
    d_my, d_te = decide(P_my, thr), decide(P_te, thr)
    kw = y_my < 8
    hit = np.mean(d_my[kw] == y_my[kw])
    fa_my = np.mean(d_my[~kw] >= 0)
    neg_te = y_te >= 8
    fa_te = np.mean(d_te[neg_te] >= 0)
    kw_te = np.mean(d_te[~neg_te] == y_te[~neg_te])
    print(f'{name:10s} esik {thr:.2f} | senin komutlarin tanindi: %{100*hit:5.1f} | '
          f'senin "bilinmeyen" yanlis kabul: %{100*fa_my:5.1f} | veri seti yanlis kabul: %{100*fa_te:5.1f} | veri seti komut: %{100*kw_te:5.1f}')

P_my_new, P_te_new = probs_tflite(new_model, X_my), probs_tflite(new_model, X_te)
old = open('kws_int8_v2.tflite', 'rb').read()
pad = lambda P: np.concatenate([P, np.zeros((len(P), 2), np.float32)], 1)   # eski model 8 sinifli
P_my_old, P_te_old = pad(probs_tflite(old, X_my)), pad(probs_tflite(old, X_te))

report('ESKI (v2)', P_my_old, P_te_old, 0.75)
for thr in [0.50, 0.60, 0.70, 0.75, 0.80, 0.90]:
    report('YENI', P_my_new, P_te_new, thr)

## 10. Senin test kayıtlarında tek tek sonuçlar (eşik 0,75)

In [ ]:
d_old, d_new = decide(P_my_old, 0.75), decide(P_my_new, 0.75)
nm = lambda d: LABELS[d] if d >= 0 else '-'
for k in range(len(y_my)):
    print(f'{LABELS[y_my[k]]:10s} -> eski: {nm(d_old[k]):6s} yeni: {nm(d_new[k]):6s} (yeni en iyi: {LABELS[P_my_new[k].argmax()]} %{100*P_my_new[k].max():.0f})')

## 11. Yeni modeli ve güncellenmiş `mfcc_consts.h`'yi indir

In [ ]:
it = tf.lite.Interpreter(model_content=new_model); it.allocate_tensors()
s_new, zp_new = it.get_input_details()[0]['quantization']
print('yeni giris olcegi:', s_new, zp_new)
open('kws_int8_v3.tflite', 'wb').write(new_model)
h2 = re.sub(r'#define MODEL_IN_SCALE\s+[-\d.e+]+f', f'#define MODEL_IN_SCALE  {s_new:.10e}f', h)
h2 = re.sub(r'#define MODEL_IN_ZP\s+\(-?\d+\)', f'#define MODEL_IN_ZP     ({zp_new})', h2)
open('mfcc_consts.h', 'w').write(h2)
print('Sinif sirasi (firmware icin):', LABELS)
files.download('kws_int8_v3.tflite'); files.download('mfcc_consts.h')